In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib
import os

# 1. Load Processed Data
df = pd.read_csv('../data/processed/employee_attrition_processed.csv')

# 2. Define X (Features) and y (Target)
X = df.drop(columns=['Attrition_Yes'])
y = df['Attrition_Yes']

# 3. Train-Test Split (80% training, 20% testing)
# stratify=y ensures the 80/20 split keeps the same proportion of Yes/No
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Handle Imbalanced Data using SMOTE (Only on Training Data!)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"--- Data After SMOTE (Training Set) ---")
print(y_train_smote.value_counts())
print("\n" + "="*40 + "\n")

# 5. Initialize Models
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss')
}

# 6. Train and Evaluate Models
best_model = None
best_roc_auc = 0
best_model_name = ""

for name, model in models.items():
    print(f"--- Evaluating {name} ---")
    
    # Train the model
    model.fit(X_train_smote, y_train_smote)
    
    # Make predictions on the Test set
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] # Probabilities for ROC-AUC
    
    # Print Metrics
    print(classification_report(y_test, y_pred))
    roc_auc = roc_auc_score(y_test, y_prob)
    print(f"ROC-AUC Score: {roc_auc:.4f}\n")
    print("-" * 40 + "\n")
    
    # Save best model logic based on ROC-AUC
    if roc_auc > best_roc_auc:
        best_roc_auc = roc_auc
        best_model = model
        best_model_name = name

# 7. Save the Best Model for Deployment
os.makedirs('../models', exist_ok=True)
# Save both the model and the feature names so we can use them exactly in the API later
model_data = {
    'model': best_model,
    'features': list(X.columns)
}
joblib.dump(model_data, '../models/best_attrition_model.pkl')

print(f"🏆 Best Model is [{best_model_name}] and it's saved successfully in '../models/'!")

--- Data After SMOTE (Training Set) ---
Attrition_Yes
False    986
True     986
Name: count, dtype: int64


--- Evaluating Random Forest ---
              precision    recall  f1-score   support

       False       0.86      0.93      0.89       247
        True       0.37      0.21      0.27        47

    accuracy                           0.82       294
   macro avg       0.62      0.57      0.58       294
weighted avg       0.78      0.82      0.80       294

ROC-AUC Score: 0.7402

----------------------------------------

--- Evaluating XGBoost ---
              precision    recall  f1-score   support

       False       0.88      0.96      0.91       247
        True       0.56      0.30      0.39        47

    accuracy                           0.85       294
   macro avg       0.72      0.63      0.65       294
weighted avg       0.83      0.85      0.83       294

ROC-AUC Score: 0.7300

----------------------------------------

🏆 Best Model is [Random Forest] and it's saved s